# Notebook 13 — Full Pipeline Inference (Demo)

End-to-end demo: image → document class → (if invoice) extracted fields.

**Stage 1 — Classification:** `microsoft/dit-large-finetuned-rvlcdip` fine-tuned
in Notebook 10b on FATURA invoices + RVL-CDIP for other classes.  
**Stage 2 — Field extraction:** `microsoft/layoutlmv3-base` fine-tuned in
Notebook 12. Only runs when stage 1 predicts *invoice*.  
**OCR:** Tesseract (pytesseract) is used on real-world images for which no
pre-computed annotations exist. Bboxes are normalised to [0, 1000] for LayoutLMv3.

> **No generative AI** — both models are discriminative only.

## 0. Imports, paths & model loading

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pytesseract
from PIL import Image
from transformers import (
    AutoFeatureExtractor,
    AutoModelForImageClassification,
    LayoutLMv3ForTokenClassification,
    LayoutLMv3Processor,
)

# Make src/ importable from this notebook
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from pipeline import DocumentPipeline

# ── Tesseract binary (Homebrew install) ───────────────────────────────────────
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Model checkpoint paths ────────────────────────────────────────────────────
DIT_CKPT       = PROJECT_ROOT / 'models' / 'experimental' / 'dit_fatura'
LMV3_CKPT      = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
DATASET_DIR    = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
FATURA_IMG_DIR = (PROJECT_ROOT / 'data' / 'raw' / 'fatura' / 'images'
                  / 'invoices_dataset_final' / 'images')

print('DiT checkpoint :', DIT_CKPT)
print('LMv3 checkpoint:', LMV3_CKPT)

## 1. Load both models via DocumentPipeline

In [ ]:
pipeline = DocumentPipeline(
    dit_checkpoint=str(DIT_CKPT),
    lmv3_checkpoint=str(LMV3_CKPT),
    label2id_path=str(DATASET_DIR / 'label2id.json'),
    id2label_path=str(DATASET_DIR / 'id2label.json'),
    device=str(DEVICE),
)
print('Pipeline ready. Doc classes:', pipeline.doc_labels)

## 2. `process_document` — single image end-to-end

In [ ]:
def process_document(image_path):
    """
    Run the full pipeline on one image and return a structured dict.

    Returns:
      {
        'image_path': str,
        'doc_class':  str,
        'class_conf': float,         # classifier confidence
        'fields':     dict | None,   # None when doc is not an invoice
        'error':      str | None,
      }
    """
    result = {
        'image_path': str(image_path),
        'doc_class':  None,
        'class_conf': None,
        'fields':     None,
        'error':      None,
    }

    path = Path(image_path)
    if not path.exists():
        result['error'] = f'File not found: {image_path}'
        return result

    try:
        pred = pipeline.predict(str(path))
    except Exception as e:
        result['error'] = f'Pipeline error: {e}'
        return result

    result['doc_class']  = pred['doc_class']
    result['class_conf'] = pred['class_conf']
    result['fields']     = pred.get('fields')   # None for non-invoices
    result['error']      = pred.get('error')
    return result


# Quick smoke test on one FATURA image
_test_img = str(next(FATURA_IMG_DIR.glob('*.jpg')))
print('Smoke test image:', _test_img)
smoke = process_document(_test_img)
print('class :', smoke['doc_class'], f'({smoke["class_conf"]:.3f})')
print('fields:', smoke['fields'])
print('error :', smoke['error'])

## 3. Run on 5 FATURA test images

In [ ]:
import random
random.seed(42)

# Load test stems from project CSV
test_df   = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'test_fatura.csv')
test_inv  = test_df[test_df['class_name'] == 'invoice']['file_path'].tolist()
demo_imgs = random.sample(test_inv, min(5, len(test_inv)))

results = [process_document(p) for p in demo_imgs]

# Display results
rows = []
for r in results:
    fields = r['fields'] or {}
    rows.append({
        'file':            Path(r['image_path']).name,
        'class':           r['doc_class'],
        'conf':            round(r['class_conf'], 3) if r['class_conf'] else None,
        'INVOICE_NUMBER':  fields.get('INVOICE_NUMBER', ''),
        'INVOICE_DATE':    fields.get('INVOICE_DATE', ''),
        'DUE_DATE':        fields.get('DUE_DATE', ''),
        'ISSUER_NAME':     fields.get('ISSUER_NAME', ''),
        'RECIPIENT_NAME':  fields.get('RECIPIENT_NAME', ''),
        'TOTAL_AMOUNT':    fields.get('TOTAL_AMOUNT', ''),
        'error':           r['error'],
    })

df_results = pd.DataFrame(rows)
print('=== Pipeline results on 5 FATURA test invoices ===')
display(df_results)

## 4. Edge cases

In [ ]:
# ── Edge case 1: non-invoice document ────────────────────────────────────────
# Pick a non-invoice from the test set
non_inv_df = test_df[test_df['class_name'] != 'invoice'].head(1)
if len(non_inv_df):
    ni_path = non_inv_df['file_path'].iloc[0]
    ni_result = process_document(ni_path)
    print('Non-invoice test:')
    print(f'  True class : {non_inv_df["class_name"].iloc[0]}')
    print(f'  Predicted  : {ni_result["doc_class"]} ({ni_result["class_conf"]:.3f})')
    print(f'  Fields     : {ni_result["fields"]}  (expected None — extraction skipped)')

# ── Edge case 2: missing file ─────────────────────────────────────────────────
missing = process_document('/tmp/does_not_exist.jpg')
print('\nMissing file test:')
print(f'  error: {missing["error"]}')

# ── Edge case 3: blank / no-text image ───────────────────────────────────────
blank = Image.new('RGB', (595, 841), color=(255, 255, 255))
blank_path = '/tmp/blank_invoice.jpg'
blank.save(blank_path)
blank_result = process_document(blank_path)
print('\nBlank image test:')
print(f'  class  : {blank_result["doc_class"]}')
print(f'  fields : {blank_result["fields"]}')
print(f'  error  : {blank_result["error"]}')

## 5. `evaluate_pipeline` — batch evaluation

In [ ]:
def evaluate_pipeline(image_paths, true_doc_classes):
    """
    Run the pipeline on a list of images and return a classification summary.

    Parameters
    ----------
    image_paths     : list[str]  — paths to image files
    true_doc_classes: list[str]  — ground-truth class names

    Returns
    -------
    pd.DataFrame with per-image predictions + overall accuracy printed.
    """
    assert len(image_paths) == len(true_doc_classes)

    rows = []
    for path, true_cls in zip(image_paths, true_doc_classes):
        r = process_document(path)
        rows.append({
            'file':       Path(path).name,
            'true_class': true_cls,
            'pred_class': r['doc_class'] or 'ERROR',
            'conf':       round(r['class_conf'], 3) if r['class_conf'] else None,
            'correct':    r['doc_class'] == true_cls,
            'error':      r['error'],
        })

    df = pd.DataFrame(rows)
    accuracy = df['correct'].mean()
    print(f'Classification accuracy: {accuracy:.3f}  ({df["correct"].sum()}/{len(df)})')
    return df


# Evaluate on all 5 non-invoice classes in test set (up to 10 images each)
eval_paths, eval_labels = [], []
for cls in ['invoice', 'form', 'resume', 'email', 'budget']:
    cls_rows = test_df[test_df['class_name'] == cls]['file_path'].tolist()
    sample   = random.sample(cls_rows, min(10, len(cls_rows)))
    eval_paths.extend(sample)
    eval_labels.extend([cls] * len(sample))

print(f'Evaluating on {len(eval_paths)} images across all 5 classes…')
eval_df = evaluate_pipeline(eval_paths, eval_labels)
print()
display(eval_df)

## 6. Visual inspection of pipeline output

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FIELD_COLOURS = {
    'INVOICE_NUMBER': '#e41a1c',
    'INVOICE_DATE':   '#377eb8',
    'DUE_DATE':       '#ff7f00',
    'ISSUER_NAME':    '#4daf4a',
    'RECIPIENT_NAME': '#984ea3',
    'TOTAL_AMOUNT':   '#a65628',
}


def visualise_result(result, show=True):
    """Display the image with predicted class and extracted field text."""
    if result['error']:
        print('Error:', result['error'])
        return

    img = Image.open(result['image_path']).convert('RGB')
    fig, ax = plt.subplots(1, 1, figsize=(5, 7))
    ax.imshow(img)
    ax.axis('off')

    title = (f"Class: {result['doc_class']} "
             f"({result['class_conf']:.2%} conf)")
    ax.set_title(title, fontsize=10)

    if result['fields']:
        text_lines = []
        for field, value in result['fields'].items():
            colour = FIELD_COLOURS.get(field, 'black')
            text_lines.append((f'{field}: {value}', colour))
        y = 0.02
        for text, colour in text_lines:
            ax.text(0.01, y, text, transform=ax.transAxes,
                    fontsize=7, color=colour,
                    bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))
            y += 0.055

    fig.tight_layout()
    if show:
        plt.show()
    return fig


for r in results[:3]:
    visualise_result(r)

## 7. Summary

In [ ]:
print('=' * 60)
print('NOTEBOOK 13 — FULL PIPELINE DEMO COMPLETE')
print('=' * 60)
print()
print('Pipeline architecture:')
print('  Stage 1 — Document classifier (DiT)')
print('             Input:  raw image')
print('             Output: class label + confidence')
print()
print('  Stage 2 — Field extractor (LayoutLMv3)  [invoice only]')
print('             Input:  image + OCR words/bboxes')
print('             Output: INVOICE_NUMBER, INVOICE_DATE, DUE_DATE,')
print('                     ISSUER_NAME, RECIPIENT_NAME, TOTAL_AMOUNT')
print()
print('For programmatic use, import DocumentPipeline from src/pipeline.py:')
print('  from pipeline import DocumentPipeline')
print('  p = DocumentPipeline(dit_checkpoint=..., lmv3_checkpoint=...)')
print('  result = p.predict("invoice.jpg")')